## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> def t1():\n'
                                       '...     pb = {}\n'
                                       "...     ph1 = '902-555-5555'\n"
                                       "...     add_contact(pb, 'andrew', ph1)\n"
                                       "...     ph2 = get_contact(pb, 'andrew')\n"
                                       '...     return ph1 == ph2\n'
                                       '>>> t1()\n'
                                       'True',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> def t2():\n'
                                       '...     pb = {}\n'
                                       "...     ph1 = '902-555-5554'\n"
                                       "...     add_contact(pb, 'sally', ph1)\n"
                                       "...     add_contact(pb, 'george', '999-999-9999')\n"
                                       "...     save(pb, 'a.txt')\n"
                                       "...     pb2 = load_contacts('a.txt')\n"
                                       "...     ph2 = get_contact(pb2, 'sally')\n"
                                       '...     return ph1 == ph2\n'
                                       '>>> t2()\n'
                                       'True',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('13-Dictionaries', _TESTS)
check = grader.check

print("Ready. 13-Dictionaries loaded: 1 question(s), 2 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

In [ ]:
#@title Run your setup...
#@markdown When you run this block you will either see ✅ SUCCESS or ❌ ERROR <br> letting you know if the library has run correctly. <br><br>
#@markdown If there is an error in this block please email your instructor for help.

try:


    print("✅ SUCCESS: All libraries imported successfully!")
    print("✅ HW Form Library loaded and ready!")
    print("📚 Available form types:")
    print("   • forms.create_info_form() - Information collection")
    print("   • forms.create_question_form() - Q&A with model answers")
    print("   • forms.create_prediction_table() - Code output prediction")
    print("   • forms.create_validation_form() - Truth tables with validation")
    print("   • forms.add_educational_context() - Structured learning context")
    print("   • forms.add_quick_context() - Simple markdown context")
    print("🆕 NEW: All forms now support optional default values!")
    print("You're ready to start the HW!")


except ImportError as e:
    print("❌ ERROR: There was a problem importing the required libraries.")
    print("\n🆘 PLEASE EMAIL YOUR INSTRUCTOR FOR HELP")
    print("   They will help you resolve this import issue.")
    print("\n" + "="*50)
    print("🔍 Technical Details (click triangle below to expand):")

    from IPython.display import HTML, display
    display(HTML(f"""
    <details>
    <summary><strong>Click here if you're curious about the error details</strong></summary>
    <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
    ImportError: {e}
    </pre>
    <p><em>Your instructor can use these details to help diagnose the problem.</em></p>
    </details>
    """))

except Exception as e:
    print("❌ ERROR: Something unexpected happened during setup.")
    print("\n🆘 PLEASE EMAIL YOUR INSTRUCTOR FOR HELP")
    print("   Show them this error message.")
    print("\n" + "="*50)
    print("🔍 Technical Details (click triangle below to expand):")

    from IPython.display import HTML, display
    display(HTML(f"""
    <details>
    <summary><strong>Click here if you're curious about the error details</strong></summary>
    <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
    Error: {e}
    Error Type: {type(e).__name__}
    </pre>
    <p><em>Your instructor can use these details to help diagnose the problem.</em></p>
    </details>
    """))

# Homework: Dictionaries — Contact List Manager

In this homework we will create a python program to store phone numbers for our contacts. 

We will store contacts in a dictionary in our running program:

```python
phonebook = {
    "Alice Smith": "555-1234",
    "Bob Johnson": "555-9876",
    "Charlie Brown": "555-5555"
}
```

We will also allow our program to persist when it is not running by writing out the contact information to a text file that might look like this:

```ascii
Alice Smith:555-1234
Bob Johnson:555-9876
Charlie Brown:555-5555

```

one entry per line with the name:phone number

Note: Even though Co-Lab supports reading and writing files to what looks like a local file system (accessible in the left side bar via the file folder icon), the file system is not persistent, it disappears and is re-created when we leave and come back to Co-lab. We need to connect to a cloud file service to allow persistence. The following cell will connect Co-Lab to your Google Drive (prompting for a login) to allow persistence, you don't need to run this cell if you do not want to connect to google drive but your files will not persist when you revisit this page unless you save them somewhere persistent like google drive.

In [ ]:
#connect to google drive
from google.colab import drive
drive.mount('/content/gdrive')

%cd /content/gdrive/MyDrive

!cp -rf /content/tests /content/gdrive/MyDrive

## Contact list implementation


To implement the contact list notice that there are 5 functions that need to be completed:

1. `add_contact`
2. `get_contact`
3. `print_all`
4. `save`
5. `load_contacts`

The descriptions of what each of the above functions should accomplish is written in the docstring within the function code below. 

Complete each function to complete the homework. 

The suggested order to complete the functions is the order they are provided above. For example: complete the `add_contact` function first. The first 3 functions do not involve files or file input/output any interactions are with a dictionary where the key is the name of the contact and the value is their phone number. For example the `add_contact` function adds a contact to the dictionary where contacts live it does not add the contact to a file. The last 2 functions involve reading and/or writing to a file where the whole dictionary is either read from or written to file.

After you have completed the code, run the below test to ensure your code behaves as expected.

### Helpful tip
- If you change the way you store data in the file and your program keeps trying to open that file (and it is causing issues) you may need to delete the file to remove the old data and start over. The data is read and written from phonebook.txt: you can delete this file by clicking the file folder icon in the left side bar and then right-clicking phonebook.txt and selecting delete.

- Make sure your methods do not have any extra print statements beyond what they are required to have this could impact the autograding below.


_Points:_ 2

In [ ]:
# Task 1: User Interface

# --- Main Application Loop ---

def display_menu():
    """Prints the user-facing menu options."""
    print("\n--- Phonebook Manager ---")
    print("1. Add/Update Contact")
    print("2. Retrieve Contact Information")
    print("3. View All Contacts")
    print("4. Save & Exit")
    print("-------------------------")

def add_contact(phone_book, name, ph_number):
    """add a contact to the phone_book
    
        phone_book: the dictionary of name:number associations
        name: the key for the dictionary
        ph_number: the value to associate with the name (key)

        return: None
    """
    ...
    
    
def get_contact(phone_book, name):
    """
        Return the phone number for the given name or the string "No such contact" if that 
        name is not in the phone_book

        phone_book: the dictionary of name:phone number pairs
        name: the key in the phone book to retrieve the value (phone number) for

        return: the phone number (string) associated with name or "No such contact"
    """
    ...

    #placeholder code 
    return ("No such contact")

def print_all(phone_book):
    """
    print the contents of the phone_book dictionary to the console
    """
    ...

def save(phone_book, file_location):
    """
        write the contents of phone_book into the file at file_location
        over_write any existing file at that location
        and return the number of contacts written to the file

        phone_book: dictionary of name and phone number associations
        file_location: the location to write the data to

        return: The number of key:value pairs written to the file   
    """
    
    ...
    # place holder code (remove)
    return 0

def load_contacts(file_location):
    """ Load the contacts from file and returns a dictionary representing that file
        
        return: a dictionary with keys as contact names and values as phone numbers (string)
    """
    ...
    #placeholder code
    return {}

# where to store the data 
file_location = "phonebook.txt"

# this code tries to open the file and load data but if it doesn't exist 
# it just uses an empty dictionary to start
try:
    phone_book = load_contacts(file_location) # Load existing data at startup
except FileNotFoundError:
    print("phonebook.txt not yet created")
    phone_book = {}

choice = 0
while choice != '4':
    
    display_menu()
    choice = input("Enter your choice (1-4): ").strip()

    if choice == '1':
        name = input("Enter the name of the contact: ").strip()
        number = input("Enter the phone number of the contact: ").strip()
        add_contact(phone_book, name, number)
    
    elif choice == '2':
        name = input("Enter the name of the contact to retrieve: ").strip()
        contact = get_contact(phone_book, name)
        if contact != "No such contact":
            print(name, contact)
        else:
            print(contact)
    
    elif choice == '3':
        print_all(phone_book)
   
    elif choice == '4':
        print("saving contacts to file")
        cnt = save(phone_book, file_location)
        print(f"saved {cnt} contacts to file")
        print("👋 Thank you for using the Phonebook Manager. Goodbye!")
    
    else:
        print("🚫 Invalid choice. Please enter a number between 1 and 4.")



In [ ]:
grader.check("q1")

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

If you visit the link from the course notes you'll come back to the original version of this file each time.

Ensure you run all the cells in order from the top to the bottom
Check to ensure all the tests are passing - run each of the cells containing grader.check ...
After running all the cells (top to bottom) and ensuring the tests are as you want Save your file to your local drive
File->Download->Download.ipynb and then upload the .ipynb file to Moodle under the appropriate Homework submission

Note that your browswer may save your .ipynb file into your Downloads folder or other.

Download and submit the .ipynb version of this file and submit to the Moodle link for Homework submission before the deadline to ensure your grade is recorded.

If you have any issues with downloading or submitting your file just email your professor to ask for help.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
